In [5]:
# Cell 1: Install Dependencies & Imports
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn', '-q'])

import pandas as pd
import numpy as np
import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed

print('All imports successful.')

All imports successful.


In [6]:
# Cell 2: Load & Preprocess Data
df = pd.read_csv('live_data.csv')

# Name columns if the CSV has no header (written by fetch_log.py's writer.writerow)
expected_cols = [
    'timestamp', 'machine', 'cpu_usage', 'gpu_wrk_util',
    'avg_mem', 'max_mem', 'avg_gpu_wrk_mem', 'max_gpu_wrk_mem',
    'read', 'write', 'read_count', 'write_count'
]
if df.columns[0] != 'timestamp':
    df.columns = expected_cols

df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%m-%y %H:%M:%S')

# Keep only recent data
original_count = len(df)
df = df[df['timestamp'].dt.year >= 2025].copy()
print(f'Loaded {original_count} rows, kept {len(df)} rows (year >= 2025).')

features = [
    'cpu_usage', 'gpu_wrk_util', 'avg_mem', 'max_mem',
    'avg_gpu_wrk_mem', 'max_gpu_wrk_mem', 'read', 'write',
    'read_count', 'write_count'
]

scaler = MinMaxScaler()
unique_servers = df['machine'].unique()
TIME_STEPS = 10

print(f'Unique servers: {unique_servers}')

Loaded 185 rows, kept 185 rows (year >= 2025).
Unique servers: <StringArray>
['server_alibaba_04', 'server_alibaba_05', 'server_alibaba_02',
 'server_alibaba_01', 'server_alibaba_03']
Length: 5, dtype: str


In [7]:
# Cell 3: Build Time-Series Sequences per Server
all_sequences = []  # <-- Properly initialized BEFORE the loop

for mac in unique_servers:
    server_data = df[df['machine'] == mac].copy()

    min_time = server_data['timestamp'].min()
    max_time = server_data['timestamp'].max()
    time_gap = max_time - min_time

    print(f'\n--- {mac} ---')
    print(f'  First log : {min_time}')
    print(f'  Last log  : {max_time}')
    print(f'  Time span : {time_gap}')

    if time_gap.days > 2:
        print(f'  WARNING: Massive time gap detected for {mac}. Skipping this server.')
        continue

    # Resample to 1-minute intervals (average)
    server_data = server_data.set_index('timestamp')
    server_minute = server_data.resample('1min').mean(numeric_only=True)
    server_minute = server_minute.dropna()  # drop empty intervals
    print(f'  Resampled to {len(server_minute)} clean 1-minute rows.')

    if len(server_minute) > TIME_STEPS:
        data_values = server_minute[features].values
        data_scaled = scaler.fit_transform(data_values)

        # Sliding window: create overlapping 10-minute sequences
        for i in range(len(data_scaled) - TIME_STEPS):
            sequence = data_scaled[i : i + TIME_STEPS]
            all_sequences.append(sequence)
    else:
        print(f'  Not enough data for {mac} to create a {TIME_STEPS}-step window. Skipping.')

print(f'\nTotal sequences built: {len(all_sequences)}')


--- server_alibaba_04 ---
  First log : 2026-03-25 11:25:52
  Last log  : 2026-03-25 11:27:24
  Time span : 0 days 00:01:32
  Resampled to 3 clean 1-minute rows.
  Not enough data for server_alibaba_04 to create a 10-step window. Skipping.

--- server_alibaba_05 ---
  First log : 2026-03-25 11:25:52
  Last log  : 2026-03-25 11:27:20
  Time span : 0 days 00:01:28
  Resampled to 3 clean 1-minute rows.
  Not enough data for server_alibaba_05 to create a 10-step window. Skipping.

--- server_alibaba_02 ---
  First log : 2026-03-25 11:25:54
  Last log  : 2026-03-25 11:27:24
  Time span : 0 days 00:01:30
  Resampled to 3 clean 1-minute rows.
  Not enough data for server_alibaba_02 to create a 10-step window. Skipping.

--- server_alibaba_01 ---
  First log : 2026-03-25 11:25:54
  Last log  : 2026-03-25 11:27:22
  Time span : 0 days 00:01:28
  Resampled to 3 clean 1-minute rows.
  Not enough data for server_alibaba_01 to create a 10-step window. Skipping.

--- server_alibaba_03 ---
  First l

In [8]:
# Cell 4: Train / Test Split
if len(all_sequences) == 0:
    raise ValueError('No sequences were built. Make sure live_data.csv has enough rows and the timestamps are valid.')

X = np.array(all_sequences)  # shape: (num_sequences, TIME_STEPS, num_features)
print(f'Dataset shape: {X.shape}')

X_train, X_test = train_test_split(X, test_size=0.2, random_state=42, shuffle=False)
print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

ValueError: No sequences were built. Make sure live_data.csv has enough rows and the timestamps are valid.

In [ ]:
# Cell 5: Build LSTM Autoencoder, Train, and Detect Anomalies
n_timesteps = X_train.shape[1]   # 10
n_features  = X_train.shape[2]   # 10

# --- Model Definition ---
model = Sequential([
    # Encoder: compress the 10-step window into a dense representation
    LSTM(32, activation='relu', input_shape=(n_timesteps, n_features), return_sequences=False),

    # Bridge: repeat the compressed vector for each time step of the decoder
    RepeatVector(n_timesteps),

    # Decoder: reconstruct the original time series
    LSTM(32, activation='relu', return_sequences=True),
    TimeDistributed(Dense(n_features))
])

model.compile(optimizer='adam', loss='mse')
model.summary()

# --- Training ---
# The model learns to reconstruct NORMAL data.
# X_train is both input AND target (autoencoder: reconstruct what you see).
history = model.fit(
    X_train, X_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

# --- Anomaly Detection on Test Set ---
# Reconstruct the test sequences and measure the per-sample MSE.
X_test_pred = model.predict(X_test)
mse_per_sample = np.mean(np.power(X_test - X_test_pred, 2), axis=(1, 2))

# Threshold: mean + 3 standard deviations (flag outliers)
threshold = np.mean(mse_per_sample) + 3 * np.std(mse_per_sample)
anomalies  = mse_per_sample > threshold

print(f'\n=== Anomaly Detection Results ===')
print(f'Reconstruction Error  - Mean : {np.mean(mse_per_sample):.6f}')
print(f'Reconstruction Error  - Std  : {np.std(mse_per_sample):.6f}')
print(f'Anomaly Threshold            : {threshold:.6f}')
print(f'Anomalies Detected           : {anomalies.sum()} / {len(mse_per_sample)} sequences')

# Show the flagged sequences
anomaly_indices = np.where(anomalies)[0]
if len(anomaly_indices) > 0:
    print(f'\nFlagged sequence indices: {anomaly_indices}')
    for idx in anomaly_indices:
        print(f'  Sequence {idx:4d}  ->  MSE = {mse_per_sample[idx]:.6f}  [ANOMALY]')
else:
    print('\nNo anomalies detected in the test set.')